In [3]:
import pandas as pd
import numpy as np
import cv2 
import os
from skimage.feature import hog
from matplotlib import pyplot as plt

In [20]:
df=pd.read_csv('detection_results_eval.csv')
df.head()


,image_path,x,y,w,h
0,train/0/00000_00000_00000.png,3,0,55,55
1,train/0/00000_00000_00000_aug0.png,0,0,57,55
2,train/0/00000_00000_00002.png,0,0,64,64
3,train/0/00000_00000_00002_aug0.png,0,0,56,55
4,train/0/00000_00000_00003.png,0,0,59,61


In [31]:
base_path = "processed_dataset"

# keep only valid splits if موجودة في المسار
df = df[df['image_path'].str.contains("train|val|test")]

print("Total samples:", len(df))

Total samples: 14620


In [32]:
def get_split_and_label(image_path):

    parts = image_path.replace("\\", "/").split("/")

    # default values
    split = "unknown"
    label = "unknown"

    # find split safely
    if "train" in parts:
        split = "train"
    elif "val" in parts:
        split = "val"
    elif "test" in parts:
        split = "test"

    # label = folder after split if exists
    try:
        split_index = parts.index(split)
        if len(parts) > split_index + 1:
            label = parts[split_index + 1]
    except:
        pass

    return split, label

In [33]:
def get_roi(img, row):

    # missing bbox → full image
    if pd.isna(row['x']) or pd.isna(row['y']):
        return img

    x, y, w, h = map(int, [row['x'], row['y'], row['w'], row['h']])

    img_h, img_w = img.shape[:2]

    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(img_w, x + w)
    y2 = min(img_h, y + h)

    if x2 <= x1 or y2 <= y1:
        return img

    return img[y1:y2, x1:x2]

In [34]:
def extract_features(roi):

    roi = cv2.resize(roi, (64, 64))

    # ---- Color (HSV histogram)
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    hsv_hist = cv2.calcHist([hsv], [0,1,2], None,
                            [8,8,8],
                            [0,180,0,256,0,256])
    hsv_hist = cv2.normalize(hsv_hist, hsv_hist).flatten()

    # ---- Shape (area from contours)
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    area = float(sum(cv2.contourArea(c) for c in contours))

    # ---- HOG
    hog_feat = hog(gray,
                   orientations=9,
                   pixels_per_cell=(8,8),
                   cells_per_block=(2,2),
                   block_norm='L2-Hys')

    return np.hstack([hsv_hist, area, hog_feat])

In [35]:
rows = []

for i, row in df.iterrows():

    image_path = os.path.join(base_path, row['image_path'])
    img = cv2.imread(image_path)

    if img is None:
        continue

    # ROI
    roi = get_roi(img, row)

    # features
    feat = extract_features(roi)

    # split + label
    split, label = get_split_and_label(row['image_path'])

    # structured row (NO mixing issues)
    full_row = [image_path, split] + feat.tolist() + [label]

    rows.append(full_row)

    if i % 200 == 0:
        print("Processed:", i)

Processed: 0
Processed: 200
Processed: 400
Processed: 600
Processed: 800
Processed: 1000
Processed: 1200
Processed: 1400
Processed: 1600
Processed: 1800
Processed: 2000
Processed: 2200
Processed: 2400
Processed: 2600
Processed: 2800
Processed: 3000
Processed: 3200
Processed: 3400
Processed: 3600
Processed: 3800
Processed: 4000
Processed: 4200
Processed: 4400
Processed: 4600
Processed: 4800
Processed: 5000
Processed: 5200
Processed: 5400
Processed: 5600
Processed: 5800
Processed: 6000
Processed: 6200
Processed: 6400
Processed: 6600
Processed: 6800
Processed: 7000
Processed: 7200
Processed: 7400
Processed: 7600
Processed: 7800
Processed: 8000
Processed: 8200
Processed: 8400
Processed: 8600
Processed: 8800
Processed: 9000
Processed: 9200
Processed: 9400
Processed: 9600
Processed: 9800
Processed: 10000
Processed: 10200
Processed: 10400
Processed: 10600
Processed: 10800
Processed: 11000
Processed: 11200
Processed: 11400
Processed: 11600
Processed: 11800
Processed: 12000
Processed: 12200
Pro

In [36]:
num_features = len(rows[0]) - 3  # image_path + split + label

columns = (
    ["image_path", "split"] +
    [f"f{i}" for i in range(num_features)] +
    ["label"]
)

df_out = pd.DataFrame(rows, columns=columns)

# =========================
# Save final dataset
# =========================
df_out.to_csv("features.csv", index=False)

print("✅ DONE - features.csv created")
print("Shape:", df_out.shape)

✅ DONE - features.csv created
Shape: (14620, 2280)
